# v34 — Wavelet-PINF + Balanced Loader + Heuristic Init

## 핵심 아이디어

v33 의 후속. PINN 의 mean shift 꼼수를 3가지 메커니즘으로 차단:

1. **Wavelet activation** (Mexican Hat / Ricker): compact support 로 mean shift 의 gradient 동력 차단
2. **Balanced Loader** (50:50): 정상 / 위기 데이터 강제 결합 oversampling
3. **Heuristic Init** (Zhang 1992): wavelet μ, σ 를 학습 데이터 hidden 분포 quantile 로 초기화

## v33 negative result (motivation)

| Spec | P(m+\|sp+) | P(m−\|sp−) | sp_mean | 진단 |
|---|---|---|---|---|
| NLL only | 61% | **63%** | +0.034 | 균형, mean shift 없음 |
| Hinge λ=0.5 | 82% | **4.7%** ⚠⚠ | +0.28 | mean shift 폭발 |
| Tanh λ=0.5 | 95% | **38%** ⚠ | +0.90 | sp+ 99% mean shift |
| **v34 wavelet only** | **18%** ⚠ | **97%** ★ | +0.40, margin_mean −0.92 | 양극화 학습 |

## v34 (Balanced + Heuristic) 가설

Wavelet 단독으로는 양극화 발생 (위기만 학습). Balanced loader + Heuristic init 추가로:
- 정상/위기 50:50 강제 → 양쪽 사분면 모두 학습
- Wavelet μ 가 hidden 분포 quantile 에 spread → 모든 영역 cover
- 기대: P(m+|sp+) 70-90%, P(m−|sp−) 80-95% (양방향 균형)

## 폴더 구조 (Drive 업로드 후)

```
/content/drive/MyDrive/Colab Notebooks/homoestatic-v34/
├── v34_wavelet.ipynb
├── data/
│   ├── weekly_v34_train.csv
│   ├── weekly_v34_test.csv
│   ├── finra_margin_monthly.csv
│   ├── market_risk_aversion.csv
│   └── fred/{M2V, GDPC1, CPIAUCSL, WM2NS}.csv
├── sim/
│   ├── favar_flow.py             ← WaveletActivation 포함
│   └── train_favar_v34.py        ← balanced + heuristic init
└── analysis/
    └── check_sign_alignment_v34.py

## 1. GPU 확인

In [ ]:
import torch, sys
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Drive 마운트 + 작업 폴더

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
COLAB_ROOT = '/content/drive/MyDrive/Colab Notebooks/homoestatic-v34'
os.chdir(COLAB_ROOT)
!pwd && ls

## 3. 필수 파일 검증

In [ ]:
from pathlib import Path
REQUIRED = [
    'data/weekly_v34_train.csv',
    'data/weekly_v34_test.csv',
    'data/finra_margin_monthly.csv',
    'data/market_risk_aversion.csv',
    'data/fred/M2V.csv',
    'data/fred/GDPC1.csv',
    'data/fred/CPIAUCSL.csv',
    'data/fred/WM2NS.csv',
    'sim/favar_flow.py',
    'sim/train_favar_v34.py',
    'analysis/check_sign_alignment_v34.py',
]
missing = [p for p in REQUIRED if not Path(p).exists()]
if missing:
    print('MISSING:'); [print(f'  {p}') for p in missing]
else:
    print(f'모든 필수 파일 OK ({len(REQUIRED)})')

## 4. 데이터/코드 → 로컬 SSD 복사 (Drive I/O 우회)

In [ ]:
import shutil, os
LOCAL = '/content/v34_workspace'
for sub in ['data/fred', 'sim', 'analysis', 'models', 'result', 'plots']:
    os.makedirs(f'{LOCAL}/{sub}', exist_ok=True)

for f in [
    'data/weekly_v34_train.csv', 'data/weekly_v34_test.csv',
    'data/finra_margin_monthly.csv', 'data/market_risk_aversion.csv',
    'data/fred/M2V.csv', 'data/fred/GDPC1.csv',
    'data/fred/CPIAUCSL.csv', 'data/fred/WM2NS.csv',
    'sim/favar_flow.py', 'sim/train_favar_v34.py',
    'analysis/check_sign_alignment_v34.py',
]:
    src, dst = f, f'{LOCAL}/{f}'
    if os.path.exists(src):
        shutil.copy(src, dst)
    else:
        print(f'  WARN: missing {src}')
print('local copy done')
os.chdir(LOCAL)
!pwd && ls

## 5. v34 본 학습 — Wavelet-PINF + Balanced + Heuristic Init

Spec:
- L=104, P=52, K=2, batch=32 (T4 메모리 안전)
- Hinge λ=0.5 + Wavelet activation
- **Balanced Loader**: crisis 정의 sp_yoy < −0.5×σ_train (학습 데이터 내 통계만, leakage 차단)
- **Heuristic Init**: 학습 시작 전 wavelet μ, σ 를 hidden 분포 quantile 로 초기화
- max-epochs 60, patience 15

추정 약 2-3시간 (Colab T4 GPU).

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

!python -u sim/train_favar_v34.py \
    --L 104 --past-len 52 --K 2 \
    --max-epochs 60 --patience 15 --batch 32 --lr 5e-4 \
    --lambda-phys 0.5 --penalty-type hinge \
    --balanced-loader --heuristic-init \
    --crisis-threshold 0.5 \
    --tag K2_balanced_heuristic 2>&1 | \
    tee result/v34_balanced_heuristic.log

## 6. 부호 일치 측정 (전체 batch)

v34 ckpt + v33 baseline 들 비교 (전체 시험 414 윈도우 × N=10 샘플).

**소요 시간 주의**: 8 ckpt × 4140 = 33000 generation. T4 면 약 30-60분.

In [ ]:
!python -u analysis/check_sign_alignment_v34.py 2>&1
!ls -la result/sign_alignment_v34* plots/sign_alignment_v34*

In [ ]:
import pandas as pd
df = pd.read_csv('result/sign_alignment_v34.csv')
cols = ['tag', 'same_sign_ratio', 'cond_match_when_sp_pos',
        'cond_match_when_sp_neg', 'corr', 'sp_mean']
df_show = df[cols].copy()
for c in ['same_sign_ratio', 'cond_match_when_sp_pos', 'cond_match_when_sp_neg']:
    df_show[c] = (df_show[c] * 100).round(1)
df_show['corr']    = df_show['corr'].round(3)
df_show['sp_mean'] = df_show['sp_mean'].round(3)
df_show.sort_values('cond_match_when_sp_neg', ascending=False)

## 7. 빠른 단일 ckpt 측정 (학습 직후 빠른 확인용)

200 윈도우 × N=10 으로 약 5-10분.

In [ ]:
import torch, sys, numpy as np
sys.path.insert(0, 'sim')
from favar_flow import MultiStepFAVARFlow, conditional_generate_favar
from train_favar_v34 import load_windows_v33

device = torch.device('cuda')

def measure(ckpt_path, n_samples=10, n_windows=200):
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    cfg = ckpt['config']
    L, P = cfg['L'], cfg['PAST_LEN']
    target_cols = cfg['COLS_TARGET']
    sp_idx = target_cols.index('sp_return')
    m_idx  = target_cols.index('margin_chg')
    use_wavelet = cfg.get('use_wavelet', False)

    X_te, C_te, _ = load_windows_v33('data/weekly_v34_test.csv', L=L, stats=ckpt['cond_stats'])
    X_te = X_te[:n_windows]; C_te = C_te[:n_windows]

    model = MultiStepFAVARFlow(K=cfg['K'], d_cond=cfg['D_COND'], d_target=cfg['D_TARGET'],
                                d_model=cfg['D_MODEL'], n_heads=cfg['N_HEADS'],
                                n_layers=cfg['N_LAYERS'], time_reverse=False,
                                use_wavelet=use_wavelet).to(device)
    model.load_state_dict(ckpt['state_dict']); model.eval()

    sp_list, m_list = [], []
    with torch.no_grad():
        for s in range(0, len(X_te), 50):
            X = X_te[s:s+50].to(device); C = C_te[s:s+50].to(device)
            x_past = X[:, :P, :].unsqueeze(1).repeat(1, n_samples, 1, 1).reshape(-1, P, 2)
            c_rep  = C.unsqueeze(1).repeat(1, n_samples, 1, 1).reshape(-1, L, cfg['D_COND'])
            x_gen = conditional_generate_favar(model, x_past, c_rep, L=L, P=P)
            future = x_gen[:, P:, :].cpu().numpy()
            sp_list.append(future[:, :, sp_idx].sum(axis=1))
            m_list.append(future[:, :, m_idx].sum(axis=1))
    sp = np.concatenate(sp_list); m = np.concatenate(m_list)
    sp_pos = sp > 0; sp_neg = sp <= 0
    p_pos = ((sp > 0) & (m > 0)).sum() / max(sp_pos.sum(), 1)
    p_neg = ((sp <= 0) & (m <= 0)).sum() / max(sp_neg.sum(), 1)
    print(f'[{ckpt_path.split("/")[-1]}]')
    print(f'  P(m+|sp+)={p_pos*100:.1f}%, P(m-|sp-)={p_neg*100:.1f}%')
    print(f'  sp_mean={sp.mean():+.3f}, sp_std={sp.std():.3f}')
    print(f'  margin_mean={m.mean():+.3f}, margin_std={m.std():.3f}')
    print(f'  sp+ share={sp_pos.mean()*100:.1f}%')

measure('models/favar_v34_K2_balanced_heuristic_best.pt')

print('\n참고:')
print('  학습:               P(m+|sp+)=93%, P(m-|sp-)=89%, sp_mean=+0.028, margin_mean=+0.064')
print('  시험:               P(m+|sp+)=74%, P(m-|sp-)=99%, sp_mean=+0.115, margin_mean=+0.065')
print('  v33 Hinge OFF:      P(m+|sp+)=82%, P(m-|sp-)= 4.7%, sp_mean=+0.28')
print('  v33 Tanh OFF:       P(m+|sp+)=95%, P(m-|sp-)=38%,  sp_mean=+0.90')
print('  v34 wavelet only:   P(m+|sp+)=18%, P(m-|sp-)=97%,  sp_mean=+0.40, margin_mean=-0.92 (양극화)')

## 8. 결과 → Drive 동기화 + zip 다운로드

In [ ]:
import shutil, os
for sub in ['models', 'result', 'plots']:
    os.makedirs(f'{COLAB_ROOT}/{sub}', exist_ok=True)
    for f in os.listdir(f'{LOCAL}/{sub}'):
        try:
            shutil.copy(f'{LOCAL}/{sub}/{f}', f'{COLAB_ROOT}/{sub}/{f}')
        except Exception as e:
            print(f'  skip {f}: {e}')
print('drive sync done')

In [ ]:
!cd $LOCAL && zip -r /content/v34_results.zip models/favar_v34_*.pt result/*.csv result/*.json result/*.log plots/*.png
from google.colab import files
files.download('/content/v34_results.zip')

## 결과 분기

| 결과 | 평가 | 다음 |
|---|---|---|
| P(m+\|sp+) 70%+ + P(m−\|sp−) 80%+ + sp_mean ≈ 0.1 | **성공 ★** | paper writeup |
| 한쪽만 70%+ (양극화 잔존) | 부분 성공 | Phase 2: Variance Annealing 추가 |
| 둘 다 약함 (50% 근처) | 실패 | Crisis-Conditional Wavelet 또는 PDE PINN 으로 전환 |